<a href="https://colab.research.google.com/github/envirogenome/RD29A-stress-gene-analysis/blob/main/RD29A_Plant_Stress_Bioinformatics_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# تثبيت Biopython
!pip install biopython

# استيراد المكتبات
from Bio import SeqIO, Entrez

# تحديد الإيميل (مطلوب من NCBI)
Entrez.email = "maszxre246@gmail.com"

# جين مقاومة الجفاف في الأرابيدوبسيس
handle = Entrez.efetch(db="nucleotide",
                       id="NM_001084670",  # RD29A gene
                       rettype="fasta",
                       retmode="text")

record = SeqIO.read(handle, "fasta")
print(record.id)
print(record.seq[:100])  # أول 100 نيوكليوتيد

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.3 MB/s eta 0:00:00
NM_001084670.1
GTGATAAACCTCTCTGGCAATTTCTTCTTCTTCTTCTCTTTTTCTTTGCTCTGCCAAAAATGGCGAGTCTACTCGATTCGTTGACAACCAGAGGGTTCTT


In [ ]:
# 🌿 RD29A Stress Gene Analysis Pipeline
# Arabidopsis thaliana | Marker-Assisted Breeding

!pip install biopython py3Dmol requests -q

from Bio import SeqIO, Entrez
from Bio.Seq import Seq
from Bio.SeqUtils import gc_fraction
import py3Dmol, requests

Entrez.email = "your_email@gmail.com"
print("✅ Libraries ready")

✅ Libraries ready


In [ ]:
# ── Fetch Gene ──────────────────────────────
handle = Entrez.efetch(db="nucleotide",
                       id="NM_001084670",
                       rettype="fasta",
                       retmode="text")
record = SeqIO.read(handle, "fasta")
seq = str(record.seq)

print("=" * 55)
print(f"🧬 Gene     : RD29A — Drought Stress Response")
print(f"🌱 Organism : Arabidopsis thaliana")
print(f"📏 Length   : {len(seq)} bp")
print(f"📊 GC       : {round(gc_fraction(record.seq)*100, 2)}%")
print("=" * 55)
print("\n📈 Nucleotide Composition:")
for base in ['A','T','G','C']:
    n = seq.count(base)
    print(f"   {base}: {n}  ({round(n/len(seq)*100,1)}%)")

🧬 Gene     : RD29A — Drought Stress Response
🌱 Organism : Arabidopsis thaliana
📏 Length   : 2205 bp
📊 GC       : 44.17%

📈 Nucleotide Composition:
   A: 629  (28.5%)
   T: 602  (27.3%)
   G: 521  (23.6%)
   C: 453  (20.5%)


In [ ]:
def design_primers(sequence, length=20, product=500):

    def tm(p):
        return 2*(p.count('A')+p.count('T')) + \
               4*(p.count('G')+p.count('C'))

    def gc(p):
        return (p.count('G')+p.count('C'))/len(p)*100

    def valid(p):
        return 40 <= gc(p) <= 60 and 55 <= tm(p) <= 65

    fwd, fwd_pos = None, 0
    for i in range(len(sequence)-length):
        c = sequence[i:i+length]
        if valid(c):
            fwd, fwd_pos = c, i
            break

    rev = None
    region = sequence[fwd_pos+product-length : fwd_pos+product]
    for i in range(len(region)-length):
        rc = str(Seq(region[i:i+length]).reverse_complement())
        if valid(rc):
            rev = rc
            break

    return fwd, rev, fwd_pos

fwd, rev, pos = design_primers(seq)

print("🔬 Primers for Marker-Assisted Breeding")
print("=" * 55)
print(f"  Forward : 5'-{fwd}-3'")
print(f"  Reverse : 5'-{rev}-3'")
print(f"  Fwd Tm  : {2*(fwd.count('A')+fwd.count('T'))+4*(fwd.count('G')+fwd.count('C'))}°C")
print(f"  Fwd GC  : {round((fwd.count('G')+fwd.count('C'))/len(fwd)*100,1)}%")
print(f"  Product : ~500 bp")
print("\n💡 Use: PCR screening for drought-tolerant lines")

🔬 Primers for Marker-Assisted Breeding
  Forward : 5'-GTGATAAACCTCTCTGGCAA-3'
  Reverse : 5'-None-3'
  Fwd Tm  : 58°C
  Fwd GC  : 45.0%
  Product : ~500 bp

💡 Use: PCR screening for drought-tolerant lines


In [ ]:
# بديل — جلب بنية جاهزة من PDB مباشرة
view = py3Dmol.view(query='pdb:2L5S', width=800, height=500)
view.setStyle({"cartoon": {"color": "spectrum"}})
view.addSurface(py3Dmol.SAS, {"opacity": 0.2, "color": "lightblue"})
view.zoomTo()
view.show()
print("✅ Stress Response Protein — 3D Structure")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

✅ Stress Response Protein — 3D Structure


In [ ]:
print("⏳ Predicting 3D structure via ESMFold...")

response = requests.post(
    "https://api.esmatlas.com/foldSequence/v1/pdb/",
    headers={"Content-Type": "application/x-www-form-urlencoded"},
    data=str(protein)
)

pdb_data = response.text

view = py3Dmol.view(width=800, height=500)
view.addModel(pdb_data, "pdb")
view.setStyle({"cartoon": {"color": "spectrum"}})
view.addSurface(py3Dmol.SAS, {"opacity": 0.2,
                               "color": "lightblue"})
view.zoomTo()
view.show()
print("✅ RD29A Protein — 3D Structure")

⏳ Predicting 3D structure via ESMFold...


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

✅ RD29A Protein — 3D Structure


In [ ]:
protein = record.seq.translate(to_stop=True)

print("🧪 Protein Translation")
print("=" * 55)
print(f"  Length   : {len(protein)} amino acids")
print(f"  Sequence : {protein[:60]}...")
print(f"\n✅ Ready for 3D structure prediction")

🧪 Protein Translation
  Length   : 27 amino acids
  Sequence : VINLSGNFFFFFSFSLLCQKWRVYSIR...

✅ Ready for 3D structure prediction
